# 01 - Exploración de Datos
## FraudIA Claims - Sistema de Detección de Fraude en Seguros

Este notebook realiza la exploración inicial del dataset de siniestros.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

sns.set_theme(style='darkgrid')
pd.set_option('display.max_columns', 50)

print('Librerías cargadas correctamente')

## 1. Generación y Carga de Datos Sintéticos

In [ ]:
from src.ingestion.generate_synthetic import main as generate_data
generate_data()

In [ ]:
from src.ingestion.load_data import load_all_from_directory

datasets = load_all_from_directory('data/synthetic')
for name, df in datasets.items():
    print(f'{name}: {df.shape}')

In [ ]:
siniestros = datasets['siniestros']
polizas = datasets['polizas']
asegurados = datasets['asegurados']
vehiculos = datasets['vehiculos']
proveedores = datasets['proveedores']
documentos = datasets['documentos']

## 2. Exploración de Siniestros

In [ ]:
print('Shape:', siniestros.shape)
print('\nColumnas:', list(siniestros.columns))
print('\nTipos de datos:')
print(siniestros.dtypes)
print('\nValores nulos:')
print(siniestros.isnull().sum())
siniestros.head()

In [ ]:
siniestros.describe()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

siniestros['ramo'].value_counts().plot.bar(ax=axes[0,0], color='#3b82f6')
axes[0,0].set_title('Distribución por Ramo')

siniestros['cobertura'].value_counts().head(10).plot.barh(ax=axes[0,1], color='#8b5cf6')
axes[0,1].set_title('Top 10 Coberturas')

siniestros['monto_reclamado'].hist(bins=50, ax=axes[0,2], color='#f59e0b')
axes[0,2].set_title('Distribución Monto Reclamado')

siniestros['estado'].value_counts().plot.bar(ax=axes[1,0], color='#22c55e')
axes[1,0].set_title('Estado del Siniestro')

siniestros['dias_entre_ocurrencia_reporte'].hist(bins=30, ax=axes[1,1], color='#ef4444')
axes[1,1].set_title('Días entre Ocurrencia y Reporte')

siniestros['etiqueta_fraude_simulada'].value_counts().plot.pie(ax=axes[1,2], autopct='%1.1f%%', colors=['#22c55e', '#ef4444'])
axes[1,2].set_title('Distribución Etiqueta Fraude')

plt.tight_layout()
plt.savefig('data/processed/exploracion_siniestros.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Análisis Cruzado

In [ ]:
fraud_by_ramo = siniestros.groupby('ramo')['etiqueta_fraude_simulada'].mean() * 100
fraud_by_ramo.sort_values(ascending=False).plot.bar(figsize=(10,5), color='#ef4444')
plt.title('Tasa de Fraude Simulada por Ramo (%)')
plt.ylabel('Porcentaje')
plt.tight_layout()
plt.show()

In [ ]:
fig = px.box(siniestros, x='ramo', y='monto_reclamado', color='etiqueta_fraude_simulada',
             title='Montos Reclamados por Ramo y Etiqueta de Fraude',
             labels={'etiqueta_fraude_simulada': 'Fraude', 'monto_reclamado': 'Monto'})
fig.show()

In [ ]:
numeric_cols = siniestros.select_dtypes(include=[np.number]).columns
corr = siniestros[numeric_cols].corr()

plt.figure(figsize=(14, 10))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, linewidths=0.5)
plt.title('Matriz de Correlación - Variables Numéricas')
plt.tight_layout()
plt.show()

## 4. Análisis de Tablas Complementarias

In [ ]:
print('=== Asegurados ===')
print(asegurados.describe())
print('\n=== Vehículos ===')
print(vehiculos['marca'].value_counts().head(10))
print('\n=== Proveedores ===')
print(f'En lista restrictiva: {proveedores["en_lista_restrictiva"].sum()}')
print(f'Total proveedores: {len(proveedores)}')
print('\n=== Documentos ===')
print(f'Total documentos: {len(documentos)}')
print(f'Con inconsistencias: {(documentos["inconsistencia_detectada"].fillna("") != "").sum()}')

In [ ]:
print('\nExploración completada. Proceder al notebook 02 para modelado.')